# 01 · Dataset, class definitions and cleaning

**ITRI626 · Colile Sibanda**

Objective **O1**: select, document and prepare the dataset, and report
the class distribution before and after cleaning, with a reason recorded
for every image removed.

The logic lives in `src/docclf/data/`; this notebook calls it and shows
the results. Source, licence and usage conditions are in
`docs/dataset-card.md`.


In [ ]:
# Every notebook opens with the same four lines: path, config, seed, paths.
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from docclf.config.loader import load_config
from docclf.config.paths import resolve_paths
from docclf.config.seed import set_global_seed
from docclf.utils.env import detect_environment

CONFIG = ROOT / "configs" / "base.yaml"
ENV = detect_environment()
cfg = load_config(CONFIG, env=ENV)
paths = resolve_paths(cfg, ENV).ensure()
set_global_seed(cfg.seed)

f"Config: {cfg.name} | Environment: {ENV} | Seed: {cfg.seed}"

## The eight classes

Fixed by the approved brief. The loader rejects any other class set.


In [ ]:
import pandas as pd

pd.DataFrame(
    {"class_id": list(cfg.class_to_id.values()), "class_name": list(cfg.classes)}
)

## Locate the dataset and build the raw manifest

One row per page image, each carrying the `doc_group_id` the split depends on.


In [ ]:
from docclf.data.acquire import fetch_dataset
from docclf.data.manifest import build_manifest

root = fetch_dataset(cfg, paths.raw)
manifest = build_manifest(root, cfg.classes, cfg.data.group_pattern)
print(f"Raw pages: {len(manifest)}")
print(f"Source documents: {manifest['doc_group_id'].nunique()}")
manifest.head()

## Class distribution before cleaning


In [ ]:
before = manifest['class_name'].value_counts().reindex(cfg.classes)
ax = before.plot.bar(figsize=(9, 3.5), color='#4C72B0')
ax.set_ylabel('Pages')
ax.set_title('Class distribution before cleaning')
before.to_frame('count')

## Cleaning

Three reasons for removal, all recorded: the file will not open, the
page is below the size floor, or it is a near-duplicate of a page
already kept in the same class.


In [ ]:
from docclf.data.clean import clean_manifest

cleaned, report = clean_manifest(manifest, cfg)
summary = report.summary()
assert summary['kept_count'] + len(report.to_frame()) == summary['raw_count']
pd.DataFrame([summary])

In [ ]:
removals = report.to_frame()
removals.to_csv(paths.splits / 'cleaning_report.csv', index=False)
removals.groupby('reason').size().to_frame('removed')

## The per-class cap, and the distribution after cleaning

The cap adds whole documents only, so no source document is ever split by it .


In [ ]:
from docclf.data.subsample import cap_per_class

working = cap_per_class(cleaned, cfg.data.n_per_class, cfg.seed, cfg.split.group_column)
working.to_csv(paths.processed / 'working_set.csv', index=False)
after = working['class_name'].value_counts().reindex(cfg.classes)
comparison = pd.DataFrame({'before': before, 'after': after})
ax = comparison.plot.bar(figsize=(9, 3.5))
ax.set_ylabel('Pages')
ax.set_title('Class distribution before and after cleaning and capping')
comparison

## Representative pages from all eight classes


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

figure, axes = plt.subplots(2, 4, figsize=(12, 7))
for panel, class_name in zip(axes.ravel(), cfg.classes):
    row = working[working['class_name'] == class_name].iloc[0]
    with Image.open(row['path']) as image:
        panel.imshow(image.convert('L'), cmap='gray')
    panel.set_title(class_name)
    panel.axis('off')
figure.suptitle('One representative page per class')
figure.tight_layout()
figure.savefig(paths.figures / 'class_examples.png', dpi=150)